In [1]:
import pandas as pd

In [ ]:
# removed hard-coded API key


In [5]:
pip install google-generativeai

  Using cached google_generativeai-0.8.6-py3-none-any.whl.metadata (3.9 kB)
  Using cached google_ai_generativelanguage-0.6.15-py3-none-any.whl.metadata (5.7 kB)
Using cached google_generativeai-0.8.6-py3-none-any.whl (155 kB)
Using cached google_ai_generativelanguage-0.6.15-py3-none-any.whl (1.3 MB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [10]:
pip install openai


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip



   ---------------------------------------- 0.0/1.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.2 MB ? eta -:--:--
   --------- ------------------------------ 0.3/1.2 MB ? eta -:--:--
   --------- ------------------------------ 0.3/1.2 MB ? eta -:--:--
   ------------------ --------------------- 0.5/1.2 MB 578.7 kB/s eta 0:00:02
   ------------------ --------------------- 0.5/1.2 MB 578.7 kB/s eta 0:00:02
   --------------------------- ------------ 0.8/1.2 MB 699.0 kB/s eta 0:00:01
   ---------------------------------------- 1.2/1.2 MB 781.2 kB/s eta 0:00:00


In [ ]:
# removed hard-coded API key


In [2]:
data=pd.read_csv(r"C:\Users\King\Downloads\STEAM_GAMES.CSV")
data

,Unnamed: 0,game_name,app_id,review_text,review_length,hours_played,review_date,owners,developers,publishers,genres,platforms,categories,release_date,price
0,0,Counter-Strike: Global Offensive,730,pucajjj bam bam,15,197.216667,2026-03-02 01:55:35,"100,000,000 .. 200,000,000",['Valve'],['Valve'],"['Action', 'Free To Play']","['windows', 'linux']","['Multi-player', 'Cross-Platform Multiplayer',...","Aug 21, 2012",NaN
1,1,Counter-Strike: Global Offensive,730,YES,3,22.850000,2026-03-02 01:39:35,"100,000,000 .. 200,000,000",['Valve'],['Valve'],"['Action', 'Free To Play']","['windows', 'linux']","['Multi-player', 'Cross-Platform Multiplayer',...","Aug 21, 2012",NaN
2,2,Counter-Strike: Global Offensive,730,its pretty fun,14,24.150000,2026-03-02 01:32:54,"100,000,000 .. 200,000,000",['Valve'],['Valve'],"['Action', 'Free To Play']","['windows', 'linux']","['Multi-player', 'Cross-Platform Multiplayer',...","Aug 21, 2012",NaN
3,3,Counter-Strike: Global Offensive,730,awsone,6,264.666667,2026-03-02 01:30:30,"100,000,000 .. 200,000,000",['Valve'],['Valve'],"['Action', 'Free To Play']","['windows', 'linux']","['Multi-player', 'Cross-Platform Multiplayer',...","Aug 21, 2012",NaN
4,4,Counter-Strike: Global Offensive,730,s,1,136.316667,2026-03-02 01:30:21,"100,000,000 .. 200,000,000",['Valve'],['Valve'],"['Action', 'Free To Play']","['windows', 'linux']","['Multi-player', 'Cross-Platform Multiplayer',...","Aug 21, 2012",NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1995,1995,Wallpaper Engine,431960,Simply goated,13,3.616667,2026-02-28 21:36:51,"20,000,000 .. 50,000,000",['Wallpaper Engine Team'],['Wallpaper Engine Team'],"['Casual', 'Indie', 'Animation & Modeling', 'D...",['windows'],"['Steam Achievements', 'Steam Trading Cards', ...","Nov 16, 2018",499.0
1996,1996,Wallpaper Engine,431960,"""it cool""",9,0.183333,2026-02-28 21:30:32,"20,000,000 .. 50,000,000",['Wallpaper Engine Team'],['Wallpaper Engine Team'],"['Casual', 'Indie', 'Animation & Modeling', 'D...",['windows'],"['Steam Achievements', 'Steam Trading Cards', ...","Nov 16, 2018",499.0
1997,1997,Wallpaper Engine,431960,good\r\n,6,3.600000,2026-02-28 21:15:45,"20,000,000 .. 50,000,000",['Wallpaper Engine Team'],['Wallpaper Engine Team'],"['Casual', 'Indie', 'Animation & Modeling', 'D...",['windows'],"['Steam Achievements', 'Steam Trading Cards', ...","Nov 16, 2018",499.0
1998,1998,Wallpaper Engine,431960,i like cat,10,59.416667,2026-02-28 20:38:56,"20,000,000 .. 50,000,000",['Wallpaper Engine Team'],['Wallpaper Engine Team'],"['Casual', 'Indie', 'Animation & Modeling', 'D...",['windows'],"['Steam Achievements', 'Steam Trading Cards', ...","Nov 16, 2018",499.0


In [ ]:
sample_df = data.sample(200, random_state=42).copy()

In [ ]:
from openai import OpenAI
import os
import pandas as pd
import time
from tqdm import tqdm

tqdm.pandas()

client = OpenAI(
    api_key=os.environ.get("OPENAI_API_KEY", ""),
    base_url="https://api.groq.com/openai/v1"
)

sample_df = data.sample(200, random_state=42).copy()

def get_sentiment_groq(text):
    prompt = f"""
You are a highly strict sentiment annotator for Steam game reviews.

Your job is classification into ONLY 3 labels:
positive, negative, neutral

-------------------------
CRITICAL RULES:
-------------------------
1. NEVER default to neutral.
2. neutral is ONLY for completely meaningless text OR no opinion at all.
3. If ANY sentiment hint exists → choose positive or negative.
4. gaming slang MUST be interpreted:
   - fun, enjoyable, addictive, amazing, good, great, insane, love → positive
   - lag, bug, crash, broken, trash, unplayable, pay-to-win → negative

5. SHORT TEXT RULE:
   If text is short (<= 5 words), STILL try to infer sentiment.

6. OUTPUT RULE:
   Return ONLY one word:
   positive OR negative OR neutral

-------------------------
EXAMPLES:
-------------------------
"this game is fun" → positive
"full of bugs" → negative
"engine" → neutral
"i love it" → positive
"trash game" → negative

-------------------------
REVIEW:
{text}
"""

    try:
        response = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[
                {"role": "user", "content": prompt}
            ],
            temperature=0
        )

        time.sleep(0.5)
        return response.choices[0].message.content.strip().lower()

    except Exception as e:
        return "neutral"



In [ ]:
from openai import OpenAI
import os
import pandas as pd
import time
from tqdm import tqdm

tqdm.pandas()

# =========================
# GROQ CLIENT
# =========================
client = OpenAI(
    api_key=os.environ.get("OPENAI_API_KEY", ""),
    base_url="https://api.groq.com/openai/v1"
)

# =========================
# SAMPLE DATA
# =========================
sample_df = data.sample(200, random_state=42).copy()

# =========================
# STRONG SENTIMENT PROMPT
# =========================
def get_sentiment_groq(text):

    prompt = f"""
You are a sentiment annotation system for Steam game reviews.

You MUST classify sentiment into exactly ONE label:

LABELS:
- positive
- negative
- neutral

========================
STRICT DECISION RULES:
========================

1. NEVER default to neutral.
2. neutral is ONLY for:
   - meaningless text (e.g. "engine", "ok", random words)
   - no opinion at all

3. If ANY emotion exists → MUST choose positive or negative.

4. Interpret gaming slang correctly:
   POSITIVE SIGNALS:
   fun, good, great, amazing, love, insane, addictive, enjoyable, awesome, nice

   NEGATIVE SIGNALS:
   lag, bug, crash, broken, trash, bad, boring, unplayable, pay-to-win, worst

5. Even VERY short text MUST be classified if possible.

6. If mixed sentiment exists → choose the dominant one.

========================
EXAMPLES:
========================
"this game is fun" → positive
"full of bugs and lag" → negative
"i love it" → positive
"trash game" → negative
"engine" → neutral
"not bad actually fun" → positive
"game is ok" → neutral

========================
REVIEW:
{text}
========================

Return ONLY one word:
positive OR negative OR neutral
"""

    try:
        response = client.chat.completions.create(
            model="llama-3.1-8b-instant",
            messages=[
                {"role": "user", "content": prompt}
            ],
            temperature=0,
            max_tokens=5
        )

        time.sleep(0.2)  # reduced delay for speed

        label = response.choices[0].message.content.strip().lower()

        # safety cleanup (very important)
        if "positive" in label:
            return "positive"
        elif "negative" in label:
            return "negative"
        elif "neutral" in label:
            return "neutral"
        else:
            return "neutral"

    except Exception as e:
        print("Error:", e)
        return "neutral"


# =========================
# APPLY WITH PROGRESS BAR
# =========================
sample_df["label_1"] = sample_df["review_text"].astype(str).progress_apply(get_sentiment_groq)

# =========================
# CHECK DISTRIBUTION
# =========================
print(sample_df["label_1"].value_counts())
sample_df.head()

  0%|          | 0/200 [00:00<?, ?it/s]

100%|██████████| 200/200 [10:17<00:00,  3.09s/it]

label_1
positive    107
negative     63
neutral      30
Name: count, dtype: int64


,Unnamed: 0,game_name,app_id,review_text,review_length,hours_played,review_date,owners,developers,publishers,genres,platforms,categories,release_date,price,label_1
1860,1860,Path of Exile 2,2694490,`,1,10.600000,2026-02-26 23:50:27,"20,000,000 .. 50,000,000",['Grinding Gear Games'],['Grinding Gear Games'],"['Action', 'Adventure', 'Massively Multiplayer...",['windows'],"['Single-player', 'Multi-player', 'MMO', 'Co-o...","Dec 6, 2024",2999.0,neutral
353,353,Palworld,1623730,incredibly fun and great replayability (i thin...,79,92.016667,2026-02-28 14:33:41,"50,000,000 .. 100,000,000",['Pocketpair'],['Pocketpair'],"['Action', 'Adventure', 'Indie', 'RPG', 'Early...",['windows'],"['Single-player', 'Multi-player', 'Co-op', 'On...","Jan 18, 2024",2999.0,positive
1333,1333,Monster Hunter Wilds,2246340,I Monster my Hunter till I Wilds,32,113.883333,2026-03-01 10:51:44,"20,000,000 .. 50,000,000","['CAPCOM Co., Ltd.']","['CAPCOM Co., Ltd.']","['Action', 'Adventure', 'RPG']",['windows'],"['Single-player', 'Multi-player', 'Co-op', 'On...","Feb 27, 2025",6999.0,negative
905,905,Left 4 Dead 2,550,This is a really good game if you love Zombie ...,74,33.550000,2026-03-01 23:18:17,"50,000,000 .. 100,000,000",['Valve'],['Valve'],['Action'],"['windows', 'linux']","['Single-player', 'Multi-player', 'PvP', 'Onli...","Nov 16, 2009",999.0,positive
1289,1289,War Thunder,236390,love the game but it pisses me the ♥♥♥♥ off un...,65,224.633333,2024-01-15 07:27:11,"20,000,000 .. 50,000,000",['Gaijin Entertainment'],['Gaijin Network Ltd'],"['Action', 'Massively Multiplayer', 'Simulatio...","['windows', 'mac', 'linux']","['Single-player', 'Multi-player', 'MMO', 'PvP'...","Aug 15, 2013",NaN,negative


In [19]:
sample_df

,Unnamed: 0,game_name,app_id,review_text,review_length,hours_played,review_date,owners,developers,publishers,genres,platforms,categories,release_date,price,label_1
1860,1860,Path of Exile 2,2694490,`,1,10.600000,2026-02-26 23:50:27,"20,000,000 .. 50,000,000",['Grinding Gear Games'],['Grinding Gear Games'],"['Action', 'Adventure', 'Massively Multiplayer...",['windows'],"['Single-player', 'Multi-player', 'MMO', 'Co-o...","Dec 6, 2024",2999.0,neutral
353,353,Palworld,1623730,incredibly fun and great replayability (i thin...,79,92.016667,2026-02-28 14:33:41,"50,000,000 .. 100,000,000",['Pocketpair'],['Pocketpair'],"['Action', 'Adventure', 'Indie', 'RPG', 'Early...",['windows'],"['Single-player', 'Multi-player', 'Co-op', 'On...","Jan 18, 2024",2999.0,positive
1333,1333,Monster Hunter Wilds,2246340,I Monster my Hunter till I Wilds,32,113.883333,2026-03-01 10:51:44,"20,000,000 .. 50,000,000","['CAPCOM Co., Ltd.']","['CAPCOM Co., Ltd.']","['Action', 'Adventure', 'RPG']",['windows'],"['Single-player', 'Multi-player', 'Co-op', 'On...","Feb 27, 2025",6999.0,neutral
905,905,Left 4 Dead 2,550,This is a really good game if you love Zombie ...,74,33.550000,2026-03-01 23:18:17,"50,000,000 .. 100,000,000",['Valve'],['Valve'],['Action'],"['windows', 'linux']","['Single-player', 'Multi-player', 'PvP', 'Onli...","Nov 16, 2009",999.0,positive
1289,1289,War Thunder,236390,love the game but it pisses me the ♥♥♥♥ off un...,65,224.633333,2024-01-15 07:27:11,"20,000,000 .. 50,000,000",['Gaijin Entertainment'],['Gaijin Network Ltd'],"['Action', 'Massively Multiplayer', 'Simulatio...","['windows', 'mac', 'linux']","['Single-player', 'Multi-player', 'MMO', 'PvP'...","Aug 15, 2013",NaN,negative
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
462,462,Team Fortress 2,440,the OG. The big chunga. This was one of the bi...,94,103.883333,2026-02-20 16:00:58,"50,000,000 .. 100,000,000",['Valve'],['Valve'],"['Action', 'Free To Play']","['windows', 'linux']","['Multi-player', 'Cross-Platform Multiplayer',...","Oct 10, 2007",NaN,positive
1105,1105,Lost Ark,1599340,I'm so grateful I quit playing in June of 2022...,319,924.116667,2026-02-22 04:17:29,"50,000,000 .. 100,000,000",['Smilegate RPG'],['Amazon Game Studios'],"['Action', 'Adventure', 'Massively Multiplayer...",['windows'],"['Single-player', 'Multi-player', 'MMO', 'PvP'...","Feb 11, 2022",NaN,negative
855,855,Grand Theft Auto V Legacy,271590,best game,9,31.683333,2026-03-01 07:10:12,"50,000,000 .. 100,000,000",['Rockstar North'],['Rockstar Games'],"['Action', 'Adventure']",['windows'],"['Single-player', 'Multi-player', 'PvP', 'Onli...","Apr 13, 2015",NaN,positive
693,693,New World: Aeternum,1063730,I started a few times over since the story cha...,592,1342.783333,2026-02-08 09:09:30,"50,000,000 .. 100,000,000",['Amazon Game Studios'],['Amazon Game Studios'],"['Action', 'Adventure', 'Massively Multiplayer...",['windows'],"['Multi-player', 'MMO', 'PvP', 'Online PvP', '...","Sep 28, 2021",NaN,negative


In [18]:
data["review_text"].loc[905]

'This is a really good game if you love Zombie Apocolypse games! Good luck,'

In [70]:
sample_df["label_1"].value_counts()

label_1
positive    107
negative     63
neutral      30
Name: count, dtype: int64

The history saving thread hit an unexpected error (OperationalError('database or disk is full')).History will not be written to the database.


In [71]:
sample_df[sample_df["label_1"]=="negative"]["review_text"].to_frame()

,review_text
1333,I Monster my Hunter till I Wilds
1289,love the game but it pisses me the ♥♥♥♥ off un...
56,valve pls anti cheat
584,cod slop
1852,LOOOOVVVVEEEEEDDDDDDDD POE1!! \n\nBut man this...
...,...
1990,i dont like men
1190,amazon sucks for man reason including this ces...
602,"Game is dead now, but the leveling felt good, ..."
1105,I'm so grateful I quit playing in June of 2022...


In [72]:
sample_df[sample_df["label_1"]=="positive"]["review_text"].to_frame()

,review_text
353,incredibly fun and great replayability (i thin...
905,This is a really good game if you love Zombie ...
1273,game good
938,Best game to play with a full group and stacke...
65,WERY GOOD GSME
...,...
1225,Leaking documents is fun 10000/10 would do again
1384,So far the best Monster Hunter Experience for ...
462,the OG. The big chunga. This was one of the bi...
855,best game


In [ ]:
# removed hard-coded API key


In [3]:
from openai import OpenAI
import os
import pandas as pd
import time
from tqdm import tqdm

tqdm.pandas()

# ===============================
# OpenRouter Client
# ===============================
client = OpenAI(
    api_key=os.environ.get("OPENAI_API_KEY", ""),
    base_url="https://openrouter.ai/api/v1"
)

MODEL_NAME = "openrouter/free"

# sample 200
sample_df = data.sample(200, random_state=42).copy()

def get_sentiment(text):
    prompt = f"""
You are an expert sentiment classifier for Steam game reviews.

Gaming sentiment rules:
- fun, amazing, replayability, addictive = positive
- lag, bug, crash, trash, pay-to-win = negative
- neutral ONLY if no opinion exists

IMPORTANT:
Do NOT overuse neutral.
Always choose positive or negative if sentiment exists.

Review:
{text}

Return ONLY one word:
positive, negative, or neutral
"""

    try:
        response = client.chat.completions.create(
            model=MODEL_NAME,
            messages=[
                {"role": "user", "content": prompt}
            ],
            temperature=0,
            max_tokens=5
        )

        time.sleep(0.3)
        return response.choices[0].message.content.strip().lower()

    except Exception as e:
        print("Error:", e)
        return "neutral"

sample_df["label_2"] = (
    sample_df["review_text"]
    .astype(str)
    .progress_apply(get_sentiment)
)

print(sample_df["label_2"].value_counts())
sample_df.head()

  1%|          | 2/200 [00:03<05:39,  1.72s/it]

Error: 'NoneType' object has no attribute 'strip'


  2%|▏         | 4/200 [00:14<13:11,  4.04s/it]

Error: 'NoneType' object has no attribute 'strip'


  4%|▎         | 7/200 [00:23<10:36,  3.30s/it]

...

100%|██████████| 200/200 [13:16<00:00,  4.65s/it]

Error: Error code: 429 - {'error': {'message': 'Provider returned error', 'code': 429, 'metadata': {'raw': 'cognitivecomputations/dolphin-mistral-24b-venice-edition:free is temporarily rate-limited upstream. Please retry shortly, or add your own key to accumulate your rate limits: https://openrouter.ai/settings/integrations', 'provider_name': 'Venice', 'is_byok': False}}, 'user_id': 'user_REDACTED'}


100%|██████████| 200/200 [13:22<00:00,  4.01s/it]

Error: Error code: 429 - {'error': {'message': 'Rate limit exceeded: free-models-per-day. Add 10 credits to unlock 1000 free model requests per day', 'code': 429, 'metadata': {'headers': {'X-RateLimit-Limit': '50', 'X-RateLimit-Remaining': '0', 'X-RateLimit-Reset': '1775865600000'}, 'provider_name': None}}, 'user_id': 'user_REDACTED'}
label_2
neutral                          181
positive                          12
negative                           4
we need to classify classify       1
we need to classify:               1
we need to classify sentiment      1
Name: count, dtype: int64


,Unnamed: 0,game_name,app_id,review_text,review_length,hours_played,review_date,owners,developers,publishers,genres,platforms,categories,release_date,price,label_2
1860,1860,Path of Exile 2,2694490,`,1,10.600000,2026-02-26 23:50:27,"20,000,000 .. 50,000,000",['Grinding Gear Games'],['Grinding Gear Games'],"['Action', 'Adventure', 'Massively Multiplayer...",['windows'],"['Single-player', 'Multi-player', 'MMO', 'Co-o...","Dec 6, 2024",2999.0,neutral
353,353,Palworld,1623730,incredibly fun and great replayability (i thin...,79,92.016667,2026-02-28 14:33:41,"50,000,000 .. 100,000,000",['Pocketpair'],['Pocketpair'],"['Action', 'Adventure', 'Indie', 'RPG', 'Early...",['windows'],"['Single-player', 'Multi-player', 'Co-op', 'On...","Jan 18, 2024",2999.0,positive
1333,1333,Monster Hunter Wilds,2246340,I Monster my Hunter till I Wilds,32,113.883333,2026-03-01 10:51:44,"20,000,000 .. 50,000,000","['CAPCOM Co., Ltd.']","['CAPCOM Co., Ltd.']","['Action', 'Adventure', 'RPG']",['windows'],"['Single-player', 'Multi-player', 'Co-op', 'On...","Feb 27, 2025",6999.0,neutral
905,905,Left 4 Dead 2,550,This is a really good game if you love Zombie ...,74,33.550000,2026-03-01 23:18:17,"50,000,000 .. 100,000,000",['Valve'],['Valve'],['Action'],"['windows', 'linux']","['Single-player', 'Multi-player', 'PvP', 'Onli...","Nov 16, 2009",999.0,we need to classify classify
1289,1289,War Thunder,236390,love the game but it pisses me the ♥♥♥♥ off un...,65,224.633333,2024-01-15 07:27:11,"20,000,000 .. 50,000,000",['Gaijin Entertainment'],['Gaijin Network Ltd'],"['Action', 'Massively Multiplayer', 'Simulatio...","['windows', 'mac', 'linux']","['Single-player', 'Multi-player', 'MMO', 'PvP'...","Aug 15, 2013",NaN,negative


In [7]:
from openai import OpenAI
import os
import pandas as pd
import time
from tqdm import tqdm

tqdm.pandas()

client = OpenAI(
    api_key=os.environ.get("OPENAI_API_KEY", ""),
    base_url="https://openrouter.ai/api/v1"
)

MODEL_NAME = "meta-llama/llama-3.3-70b-instruct:free"
sample_df = data.sample(200, random_state=42).copy()

def normalize_label(label):
    label = str(label).lower().strip()

    if "positive" in label:
        return "positive"
    elif "negative" in label:
        return "negative"
    elif "neutral" in label:
        return "neutral"
    return "neutral"


def get_sentiment(text):
    prompt = f"""
You are an expert classifier for Steam game reviews.

Rules:
- praise, fun, amazing, love, replayability => positive
- lag, crash, bug, boring, trash => negative
- neutral only if there is no opinion

Review:
{text}

Return only:
positive
negative
neutral
"""

    try:
        response = client.chat.completions.create(
            model=MODEL_NAME,
            messages=[
                {"role": "system", "content": "You are a sentiment classifier."},
                {"role": "user", "content": prompt}
            ],
            temperature=0,
            max_tokens=3
        )

        output = response.choices[0].message.content
        time.sleep(0.5)

        return normalize_label(output)

    except Exception as e:
        print("ERROR =>", e)
        return "neutral"

sample_df["label_2"] = (
    sample_df["review_text"]
    .astype(str)
    .progress_apply(get_sentiment)
)

print(sample_df["label_2"].value_counts())

  1%|          | 2/200 [00:03<05:22,  1.63s/it]

ERROR => Error code: 429 - {'error': {'message': 'Provider returned error', 'code': 429, 'metadata': {'raw': 'meta-llama/llama-3.3-70b-instruct:free is temporarily rate-limited upstream. Please retry shortly, or add your own key to accumulate your rate limits: https://openrouter.ai/settings/integrations', 'provider_name': 'Venice', 'is_byok': False}}, 'user_id': 'user_REDACTED'}


  2%|▏         | 3/200 [00:05<06:13,  1.89s/it]

ERROR => Error code: 429 - {'error': {'message': 'Provider returned error', 'code': 429, 'metadata': {'raw': 'meta-llama/llama-3.3-70b-instruct:free is temporarily rate-limited upstream. Please retry shortly, or add your own key to accumulate your rate limits: https://openrouter.ai/settings/integrations', 'provider_name': 'Venice', 'is_byok': False}}, 'user_id': 'user_REDACTED'}


  2%|▏         | 4/200 [00:07<06:33,  2.01s/it]

...

ERROR => Error code: 429 - {'error': {'message': 'Rate limit exceeded: limit_rpm/meta-llama/llama-3.3-70b-instruct/839b2e30-a1b4-4974-b980-3e534b5873b1. High demand for meta-llama/llama-3.3-70b-instruct:free on OpenRouter - limited to 8 requests per minute. Please retry shortly.', 'code': 429, 'metadata': {'headers': {'X-RateLimit-Limit': '8', 'X-RateLimit-Remaining': '0', 'X-RateLimit-Reset': '1775831820000'}, 'provider_name': None}}, 'user_id': 'user_REDACTED'}


100%|██████████| 200/200 [06:23<00:00,  1.84s/it]

ERROR => Error code: 429 - {'error': {'message': 'Rate limit exceeded: limit_rpm/meta-llama/llama-3.3-70b-instruct/839b2e30-a1b4-4974-b980-3e534b5873b1. High demand for meta-llama/llama-3.3-70b-instruct:free on OpenRouter - limited to 8 requests per minute. Please retry shortly.', 'code': 429, 'metadata': {'headers': {'X-RateLimit-Limit': '8', 'X-RateLimit-Remaining': '0', 'X-RateLimit-Reset': '1775831820000'}, 'provider_name': None}}, 'user_id': 'user_REDACTED'}


100%|██████████| 200/200 [06:25<00:00,  1.93s/it]

ERROR => Error code: 429 - {'error': {'message': 'Rate limit exceeded: limit_rpm/meta-llama/llama-3.3-70b-instruct/839b2e30-a1b4-4974-b980-3e534b5873b1. High demand for meta-llama/llama-3.3-70b-instruct:free on OpenRouter - limited to 8 requests per minute. Please retry shortly.', 'code': 429, 'metadata': {'headers': {'X-RateLimit-Limit': '8', 'X-RateLimit-Remaining': '0', 'X-RateLimit-Reset': '1775831820000'}, 'provider_name': None}}, 'user_id': 'user_REDACTED'}
label_2
neutral    200
Name: count, dtype: int64


In [9]:
pip install vaderSentiment

Note: you may need to restart the kernel to use updated packages.Collecting vaderSentiment




[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [63]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

vader = SentimentIntensityAnalyzer()

def get_sentiment_vader(text):
    score = vader.polarity_scores(str(text))['compound']

    if score >= 0.05:
        return "positive"
    elif score <= -0.05:
        return "negative"
    else:
        return "neutral"


sample_df["label_2"] = sample_df["review_text"].progress_apply(get_sentiment_vader)

100%|██████████| 200/200 [00:00<00:00, 1790.29it/s]


In [64]:
from textblob import TextBlob

def get_sentiment_textblob(text):
    polarity = TextBlob(str(text)).sentiment.polarity

    if polarity > 0.1:
        return "positive"
    elif polarity < -0.1:
        return "negative"
    else:
        return "neutral"
    

sample_df["label_3"] = sample_df["review_text"].progress_apply(get_sentiment_textblob)

100%|██████████| 200/200 [00:00<00:00, 1176.51it/s]


In [ ]:
from tqdm import tqdm
tqdm.pandas()

# sample
sample_df = data.sample(200, random_state=42).copy()

# # Groq (انت عندك function جاهز)
sample_df["label_1"] = sample_df["review_text"].progress_apply(get_sentiment_groq)

# VADER


# TextBlob


sample_df.head()

100%|██████████| 200/200 [00:00<00:00, 4807.75it/s]


,Unnamed: 0,game_name,app_id,review_text,review_length,hours_played,review_date,owners,developers,publishers,genres,platforms,categories,release_date,price,label_1,label_2,label_3
1860,1860,Path of Exile 2,2694490,`,1,10.600000,2026-02-26 23:50:27,"20,000,000 .. 50,000,000",['Grinding Gear Games'],['Grinding Gear Games'],"['Action', 'Adventure', 'Massively Multiplayer...",['windows'],"['Single-player', 'Multi-player', 'MMO', 'Co-o...","Dec 6, 2024",2999.0,"i'm ready to annotate the review. however, it ...",neutral,neutral
353,353,Palworld,1623730,incredibly fun and great replayability (i thin...,79,92.016667,2026-02-28 14:33:41,"50,000,000 .. 100,000,000",['Pocketpair'],['Pocketpair'],"['Action', 'Adventure', 'Indie', 'RPG', 'Early...",['windows'],"['Single-player', 'Multi-player', 'Co-op', 'On...","Jan 18, 2024",2999.0,neutral,positive,positive
1333,1333,Monster Hunter Wilds,2246340,I Monster my Hunter till I Wilds,32,113.883333,2026-03-01 10:51:44,"20,000,000 .. 50,000,000","['CAPCOM Co., Ltd.']","['CAPCOM Co., Ltd.']","['Action', 'Adventure', 'RPG']",['windows'],"['Single-player', 'Multi-player', 'Co-op', 'On...","Feb 27, 2025",6999.0,neutral,neutral,neutral
905,905,Left 4 Dead 2,550,This is a really good game if you love Zombie ...,74,33.550000,2026-03-01 23:18:17,"50,000,000 .. 100,000,000",['Valve'],['Valve'],['Action'],"['windows', 'linux']","['Single-player', 'Multi-player', 'PvP', 'Onli...","Nov 16, 2009",999.0,neutral,positive,positive
1289,1289,War Thunder,236390,love the game but it pisses me the ♥♥♥♥ off un...,65,224.633333,2024-01-15 07:27:11,"20,000,000 .. 50,000,000",['Gaijin Entertainment'],['Gaijin Network Ltd'],"['Action', 'Massively Multiplayer', 'Simulatio...","['windows', 'mac', 'linux']","['Single-player', 'Multi-player', 'MMO', 'PvP'...","Aug 15, 2013",NaN,neutral,positive,negative


In [65]:
sample_df

,Unnamed: 0,game_name,app_id,review_text,review_length,hours_played,review_date,owners,developers,publishers,genres,platforms,categories,release_date,price,label_1,label_2,label_3
1860,1860,Path of Exile 2,2694490,`,1,10.600000,2026-02-26 23:50:27,"20,000,000 .. 50,000,000",['Grinding Gear Games'],['Grinding Gear Games'],"['Action', 'Adventure', 'Massively Multiplayer...",['windows'],"['Single-player', 'Multi-player', 'MMO', 'Co-o...","Dec 6, 2024",2999.0,neutral,neutral,neutral
353,353,Palworld,1623730,incredibly fun and great replayability (i thin...,79,92.016667,2026-02-28 14:33:41,"50,000,000 .. 100,000,000",['Pocketpair'],['Pocketpair'],"['Action', 'Adventure', 'Indie', 'RPG', 'Early...",['windows'],"['Single-player', 'Multi-player', 'Co-op', 'On...","Jan 18, 2024",2999.0,positive,positive,positive
1333,1333,Monster Hunter Wilds,2246340,I Monster my Hunter till I Wilds,32,113.883333,2026-03-01 10:51:44,"20,000,000 .. 50,000,000","['CAPCOM Co., Ltd.']","['CAPCOM Co., Ltd.']","['Action', 'Adventure', 'RPG']",['windows'],"['Single-player', 'Multi-player', 'Co-op', 'On...","Feb 27, 2025",6999.0,negative,neutral,neutral
905,905,Left 4 Dead 2,550,This is a really good game if you love Zombie ...,74,33.550000,2026-03-01 23:18:17,"50,000,000 .. 100,000,000",['Valve'],['Valve'],['Action'],"['windows', 'linux']","['Single-player', 'Multi-player', 'PvP', 'Onli...","Nov 16, 2009",999.0,positive,positive,positive
1289,1289,War Thunder,236390,love the game but it pisses me the ♥♥♥♥ off un...,65,224.633333,2024-01-15 07:27:11,"20,000,000 .. 50,000,000",['Gaijin Entertainment'],['Gaijin Network Ltd'],"['Action', 'Massively Multiplayer', 'Simulatio...","['windows', 'mac', 'linux']","['Single-player', 'Multi-player', 'MMO', 'PvP'...","Aug 15, 2013",NaN,negative,positive,negative
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
462,462,Team Fortress 2,440,the OG. The big chunga. This was one of the bi...,94,103.883333,2026-02-20 16:00:58,"50,000,000 .. 100,000,000",['Valve'],['Valve'],"['Action', 'Free To Play']","['windows', 'linux']","['Multi-player', 'Cross-Platform Multiplayer',...","Oct 10, 2007",NaN,positive,neutral,neutral
1105,1105,Lost Ark,1599340,I'm so grateful I quit playing in June of 2022...,319,924.116667,2026-02-22 04:17:29,"50,000,000 .. 100,000,000",['Smilegate RPG'],['Amazon Game Studios'],"['Action', 'Adventure', 'Massively Multiplayer...",['windows'],"['Single-player', 'Multi-player', 'MMO', 'PvP'...","Feb 11, 2022",NaN,negative,positive,positive
855,855,Grand Theft Auto V Legacy,271590,best game,9,31.683333,2026-03-01 07:10:12,"50,000,000 .. 100,000,000",['Rockstar North'],['Rockstar Games'],"['Action', 'Adventure']",['windows'],"['Single-player', 'Multi-player', 'PvP', 'Onli...","Apr 13, 2015",NaN,positive,positive,positive
693,693,New World: Aeternum,1063730,I started a few times over since the story cha...,592,1342.783333,2026-02-08 09:09:30,"50,000,000 .. 100,000,000",['Amazon Game Studios'],['Amazon Game Studios'],"['Action', 'Adventure', 'Massively Multiplayer...",['windows'],"['Multi-player', 'MMO', 'PvP', 'Online PvP', '...","Sep 28, 2021",NaN,negative,negative,neutral


In [66]:
from statsmodels.stats.inter_rater import fleiss_kappa
import numpy as np
import pandas as pd

# labels dataframe
df = sample_df[["label_1", "label_2", "label_3"]]

def to_fleiss(row):
    return [
        (row == "positive").sum(),
        (row == "negative").sum(),
        (row == "neutral").sum()
    ]

fleiss_matrix = df.apply(to_fleiss, axis=1).tolist()


fleiss_matrix = np.array(fleiss_matrix)

score = fleiss_kappa(fleiss_matrix)

print("Fleiss Kappa =", score)

Fleiss Kappa = 0.41515214129780514


In [69]:
sample_df.to_csv(r"D:\\ground_truth.CSV")